In [ ]:
# IMPORT, CONFIG

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

from scrapers import MatchScraper, H2HScraper
from scrapers.base_scraper import BaseScraper
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

rankings = pd.read_csv("data/all_rankings2025.csv")
rankings.date = pd.to_datetime(rankings.date)

TEST_EVENT_ID = "8292"  # ESL Pro League Season 20
TEST_MATCH_INDEX = 0    # Első meccs

print(f"🎯 Konfiguráció:")
print(f"   Event ID: {TEST_EVENT_ID}")
print(f"   Match index: {TEST_MATCH_INDEX}")


In [ ]:
# 3. TARGET MATCHES SCRAPING
print("\n" + "="*60)
print("1️⃣ TARGET MATCHES SCRAPING")
print("="*60)

try:
    scraper = MatchScraper(headless=False)
    target_matches = scraper.scrape_event_matches(TEST_EVENT_ID)
    
    if target_matches.empty:
        raise ValueError("❌ Nincs target match!")
    
    # ordinal ragok eltávolítása
    target_matches["date_clean"] = target_matches["date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))

    # dátummá alakítás
    target_matches["date_parsed"] = pd.to_datetime(target_matches["date_clean"], format="%B %d %Y")

    
    print(f"✅ {len(target_matches)} meccs találva")
    display(target_matches.head(3))
    
except Exception as e:
    print(f"❌ Hiba a target matches scraping közben: {e}")

In [ ]:
# 4. MATCH KIVÁLASZTÁSA
print("\n" + "="*60)
print("2️⃣ MATCH KIVÁLASZTÁSA")
print("="*60)

if TEST_MATCH_INDEX >= len(target_matches):
    TEST_MATCH_INDEX = 0
    print(f"⚠️ Index túl nagy, első meccset használom")

selected_match = target_matches.iloc[TEST_MATCH_INDEX]

print(f"🎯 Kiválasztott meccs (index={TEST_MATCH_INDEX}):")
print(f"  Match ID:   {selected_match['match_id']}")
print(f"  Date:       {selected_match['date_parsed']}")
print(f"  Teams:      {selected_match['team_home']} vs {selected_match['team_away']}")
print(f"  Score:      {selected_match['score_home']} - {selected_match['score_away']}")
print(f"  URL:        {selected_match['link']}")

In [ ]:
# 5. MATCH H2H SCRAPING
print("\n" + "="*60)
print("3️⃣ MATCH H2H SCRAPING")
print("="*60)

try:
    match_url = selected_match['link']
    print(f"🔍 Scraping: {match_url}")
    
    scraper = H2HScraper(headless=False)
    h2h_df = scraper.scrape_match_h2h(match_url)
    
    if h2h_df.empty:
        raise ValueError("❌ H2H scraping sikertelen!")
    
    match_h2h = h2h_df.iloc[0]
    
    print(f"✅ H2H data:")
    display(match_h2h)
    
except Exception as e:
    print(f"❌ Hiba a H2H scraping közben: {e}")

In [ ]:
# 6. TEAMS EXTRACTION
print("\n" + "="*60)
print("4️⃣ TEAMS EXTRACTION")
print("="*60)

teams = {
    'home': {
        'team_id': str(match_h2h.get('home_team_id', 'unknown')),
        'team_name': match_h2h['home_team']
    },
    'away': {
        'team_id': str(match_h2h.get('away_team_id', 'unknown')), 
        'team_name': match_h2h['away_team']
    }
}

print(f"✅ Teams extracted:")
print(f"  Home: {teams['home']['team_name']} (ID: {teams['home']['team_id']})")
print(f"  Away: {teams['away']['team_name']} (ID: {teams['away']['team_id']})")

In [ ]:
# 7. TEAM HISTORY SCRAPING HELPER
print("\n" + "="*60)
print("5️⃣ TEAM HISTORY SCRAPING HELPER")
print("="*60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from scrapers.base_scraper import BaseScraper

class TeamHistoryScraper(BaseScraper):
    def scrape_team_matches(self, team_id: str, max_matches: int = 20):
        """Team match history scraping"""
        url = f"https://www.hltv.org/results?team={team_id}"
        self._init_driver()
        self.driver.get(url)

        wait = WebDriverWait(self.driver, 20)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "results-holder")))

        self._random_delay()

        matches = []
        all_sublists = self.driver.find_elements(By.CLASS_NAME, "results-sublist")
        print(f"🔍 Összesen {len(all_sublists)} results-sublist betöltve")

        for sublist in all_sublists:
            if len(matches) >= max_matches:
                break

            try:
                headline = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
                match_date = headline.replace("Results for", "").strip()
            except:
                match_date = None

            match_blocks = sublist.find_elements(By.CLASS_NAME, "result-con")
            for match in match_blocks:
                if len(matches) >= max_matches:
                    break

                try:
                    a_tag = match.find_element(By.TAG_NAME, "a")
                    match_url = a_tag.get_attribute("href")
                    match_id = match_url.split('/')[4]

                    table = a_tag.find_element(By.TAG_NAME, "table")
                    tds = table.find_elements(By.TAG_NAME, "td")

                    team1_name = tds[0].find_element(By.CLASS_NAME, "team").text.strip()
                    team2_name = tds[2].find_element(By.CLASS_NAME, "team").text.strip()

                    score_spans = tds[1].find_elements(By.TAG_NAME, "span")
                    score1 = int(score_spans[0].text.strip())
                    score2 = int(score_spans[1].text.strip())

                    team1_html = tds[0].get_attribute("innerHTML")
                    won = "team-won" in team1_html

                    try:
                        map_text = a_tag.find_element(By.CSS_SELECTOR, ".map-text").text.strip()
                    except:
                        map_text = "bo1"

                    opponent_name = team2_name if team1_name else team1_name

                    matches.append({
                        "team_id": team_id,
                        "match_id": match_id,
                        "match_date": match_date,
                        "opponent_name": opponent_name,
                        "result": "win" if won else "loss",
                        "score_for": score1,
                        "score_against": score2,
                        "map_type": map_text,
                        "link": match_url
                    })

                except Exception as e:
                    print(f"⚠️ Hiba egy meccs feldolgozásánál: {e}")
                    continue

        self.close()
        return pd.DataFrame(matches)

print("✅ TeamHistoryScraper helper kész")

In [ ]:
# 8. TEAM HISTORIES SCRAPING

print("\n" + "="*60)
print("6️⃣ TEAM HISTORIES SCRAPING")
print("="*60)

team_histories = {}

for side in ['home', 'away']:
    team_id = teams[side]['team_id']
    team_name = teams[side]['team_name']
    
    print(f"\n📈 Scraping history: {team_name} (ID: {team_id})")
    
    try:
        scraper = TeamHistoryScraper(headless=False)
        history_df = scraper.scrape_team_matches(team_id, max_matches=100)
        
        if history_df.empty:
            print(f"⚠️ Nincs history adat: {team_name}")
            team_histories[side] = pd.DataFrame()
            continue

        # ordinal ragok eltávolítása
        history_df["date_clean"] = history_df["match_date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))
        # dátummá alakítás
        history_df["date_parsed"] = pd.to_datetime(history_df["date_clean"], format="%B %d %Y")

        # csak a meccs dátuma előttiek (megelőző meccsek)
        history_df = history_df[history_df['date_parsed'] < selected_match['date_parsed']]       

        if len(history_df) < 3:
            print(f"⚠️ Nincs elég history adat: {team_name}")
            team_histories[side] = pd.DataFrame()
            continue

        team_histories[side] = history_df
        
        print(f"✅ {len(history_df)} meccs találva")

        print(f"\n📋 Megelőző 3 meccs:")
        display(history_df.head(3))
        
    except Exception as e:
        print(f"❌ Hiba a history scraping közben: {e}")

In [ ]:
# 9. HISTORICAL H2H SCRAPING

# Get rank data function
def get_rank_data(date, team_name):
    team_rankings = rankings[(rankings['date'] < date) & 
                            (rankings['team_name'] == team_name)].reset_index(drop=True)

    # Current (latest) rank & points
    rank = team_rankings.loc[0, 'rank']
    points = team_rankings.loc[0, 'points']
    # Latest rank & points change
    rank_change = team_rankings.loc[0, 'rank'] - team_rankings.loc[1, 'rank']
    points_change = team_rankings.loc[0, 'points'] - team_rankings.loc[1, 'points']

    return({'rank': rank,
            'rank_change': rank_change, 
            'points': points,
            'points_change': points_change}
            )
print("Rank data function initialized.")

print("\n" + "="*60)
print("7️⃣ HISTORICAL H2H SCRAPING")
print("="*60)

historical_h2h = {}
N_MATCHES = 3

rank_data = {}
for side in ['home', 'away']:
    team_name = teams[side]['team_name']
    team_id = teams[side]['team_id']
    date = selected_match['date_parsed']
    rank_data[side] = get_rank_data(date, team_name)
    print(f"\n🔍 {team_name} last {N_MATCHES} matches H2H scraping:")
    
    history = team_histories.get(side, pd.DataFrame())
    
    if history.empty:
        print(f"  ⚠️ Nincs history, skip")
        continue
    
    history = history[history['date_parsed'] < date].reset_index(drop=True)
    
    # Last N match URL-ek
    last_n_matches = history.head(N_MATCHES)
    
    h2h_total = []
    h2h_wins = []
    ratings = []
    rating_stds = []
    adrs = []
    adr_stds = []
    swings = []
    swing_stds = []
    maps_total = []
    maps_wins = []
    maps_picked = []
    maps_avg_score_diffs = []
    ranks_opp = []
    ranks_change = []
    rank_diffs = []
    points_opp = []
    points_change = []
    point_diffs = []
    
    for idx, match in last_n_matches.iterrows():
        match_url = match['link']
        print(f"  [{idx+1}/{N_MATCHES}] Scraping: {match['date_parsed'].date()} vs {match['opponent_name']}\n{match_url}")
        rank_data_side = get_rank_data(match['date_parsed'], team_name)

        try:
            scraper = H2HScraper(headless=False)
            hist_h2h_df = scraper.scrape_match_h2h(match_url)

            if hist_h2h_df.empty:
                raise ValueError("❌ H2H scraping sikertelen!")
            
            if str(hist_h2h_df['home_team_id'][0]) == str(team_id):
                h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                h2h_wins.append(hist_h2h_df['wins_home'][0])
                ratings.append(hist_h2h_df['home_team_avg_rating'][0])
                rating_stds.append(hist_h2h_df['home_team_std_rating'][0])
                adrs.append(hist_h2h_df['home_team_avg_ADR'][0])
                adr_stds.append(hist_h2h_df['home_team_std_ADR'][0])
                swings.append(hist_h2h_df['home_team_avg_Swing'][0])
                swing_stds.append(hist_h2h_df['home_team_std_Swing'][0])
                maps_total.append(hist_h2h_df['maps_played'][0])
                maps_wins.append(hist_h2h_df['home_maps_won'][0])
                maps_picked.append(hist_h2h_df['home_maps_picked'][0])
                maps_avg_score_diffs.append(hist_h2h_df['map_avg_score_diff'][0])
                opp = hist_h2h_df['away_team'][0]

            elif str(hist_h2h_df['away_team_id'][0]) == str(team_id):
                h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                h2h_wins.append(hist_h2h_df['wins_away'][0])
                ratings.append(hist_h2h_df['away_team_avg_rating'][0])
                rating_stds.append(hist_h2h_df['away_team_std_rating'][0])
                adrs.append(hist_h2h_df['away_team_avg_ADR'][0])
                adr_stds.append(hist_h2h_df['away_team_std_ADR'][0])
                swings.append(hist_h2h_df['away_team_avg_Swing'][0])
                swing_stds.append(hist_h2h_df['away_team_std_Swing'][0])
                maps_total.append(hist_h2h_df['maps_played'][0])
                maps_wins.append(hist_h2h_df['away_maps_won'][0])
                maps_picked.append(hist_h2h_df['maps_played'][0] - hist_h2h_df['home_maps_picked'][0])
                maps_avg_score_diffs.append( - hist_h2h_df['map_avg_score_diff'][0])
                opp = hist_h2h_df['home_team'][0]
            else:
                print("ERROR: team_id not in df")

            # HLTV Rank & points
            rank_data_opp = get_rank_data(match['date_parsed'], opp)

            ranks_opp.append(rank_data_opp['rank'])
            ranks_change.append(rank_data_opp['rank_change'])
            rank_diffs.append(rank_data_opp['rank'] - rank_data_side['rank'])
            points_opp.append(rank_data_opp['points'])
            points_change.append(rank_data_opp['points_change'])
            point_diffs.append(rank_data_side['points'] - rank_data_opp['points'])
                        
        except Exception as e:
            print(f"    ⚠️ H2H scraping hiba: {e}")
            continue
    
    # Átlagok számítása
    avg_h2h_winrate = np.sum(h2h_wins) / np.sum(h2h_total) if h2h_total else None
    avg_rating = np.mean(ratings) if ratings else None
    avg_rating_std = np.mean(rating_stds) if rating_stds else None
    avg_adr = np.mean(adrs) if adrs else None
    avg_adr_std = np.mean(adr_stds) if adr_stds else None
    avg_swing = np.mean(swings) if swings else None
    avg_swing_std = np.mean(swing_stds) if swing_stds else None
    avg_maps_total = np.mean(maps_total) if maps_total else None
    avg_map_winrate = np.sum(maps_wins) / np.sum(maps_total) if maps_wins else None
    avg_map_pickrate = np.sum(maps_picked) / np.sum(maps_total) if maps_picked else None
    avg_map_score_diff = np.mean(maps_avg_score_diffs) if maps_avg_score_diffs else None
    avg_rank_diff = np.mean(rank_diffs) if rank_diffs else None
    std_rank_diff = np.std(rank_diffs) if rank_diffs else None
    avg_point_diff = np.mean(point_diffs) if point_diffs else None
    std_point_diff = np.std(point_diffs) if point_diffs else None
    weak_opp_rank_diff = np.max(rank_diffs) if rank_diffs else None
    strong_opp_rank_diff = np.min(rank_diffs) if rank_diffs else None
    weak_opp_point_diff = np.max(point_diffs) if point_diffs else None
    strong_opp_point_diff = np.min(point_diffs) if point_diffs else None

    historical_h2h[side] = {
        'avg_h2h_winrate': avg_h2h_winrate,
        'avg_rating': avg_rating,
        'avg_rating_std': avg_rating_std,
        'avg_adr': avg_adr,
        'avg_adr_std': avg_adr_std,
        'avg_swing': avg_swing,
        'avg_swing_std': avg_swing_std,
        'avg_maps_total': avg_maps_total,
        'avg_map_winrate': avg_map_winrate,
        'avg_map_pickrate': avg_map_pickrate,
        'avg_map_score_diff': avg_map_score_diff,
        'avg_rank_diff': avg_rank_diff,
        'std_rank_diff': std_rank_diff,
        'avg_point_diff': avg_point_diff,
        'std_point_diff': std_point_diff,
        'weak_opp_rank_diff': weak_opp_rank_diff,
        'strong_opp_rank_diff': strong_opp_rank_diff,
        'weak_opp_point_diff': weak_opp_point_diff,
        'strong_opp_point_diff': strong_opp_point_diff,
        'n_matches_scraped': len(ratings)
    }

print(f"\n📊 Historical H2H stats összegzés:")
for side in ['home', 'away']:
    stats = historical_h2h[side]
    display(stats)

In [ ]:
# 10. ROLLING FEATURES SZÁMÍTÁSA
print("\n" + "="*60)
print("8️⃣ ROLLING FEATURES SZÁMÍTÁSA")
print("="*60)

ml_input_row = {}

for side in ['home', 'away']:
    team_name = teams[side]['team_name']
    print(f"\n📊 {team_name} rolling features:")
    
    history = team_histories.get(side, pd.DataFrame())
    
    if history.empty:
        print(f"  ⚠️ Nincs history adat")
        continue
    
    # Last 3 winrate
    last_3 = history.head(3)
    last_3_wr = (last_3['result'] == 'win').mean() if len(last_3) > 0 else None
    
    # Last 5 winrate  
    last_5 = history.head(5)
    last_5_wr = (last_5['result'] == 'win').mean() if len(last_5) > 0 else None
    
    # Avg scores
    last_3_avg_for = last_3['score_for'].mean() if len(last_3) > 0 else None
    last_3_avg_against = last_3['score_against'].mean() if len(last_3) > 0 else None
    
    # Current streak
    streak = 0
    if len(history) > 0:
        last_result = history.iloc[0]['result']
        for _, match in history.iterrows():
            if match['result'] == last_result:
                streak += 1
            else:
                break
        if last_result == 'loss':
            streak *= -1
    
    print(f"  Last 3 winrate:    {last_3_wr:.3f}" if last_3_wr else "  Last 3 winrate:    N/A")
    print(f"  Last 5 winrate:    {last_5_wr:.3f}" if last_5_wr else "  Last 5 winrate:    N/A")
    print(f"  Last 3 avg score:  {last_3_avg_for:.1f} - {last_3_avg_against:.1f}" if last_3_avg_for else "  Last 3 avg score:  N/A")
    print(f"  Current streak:    {streak:+d}")
    
    # Store
    ml_input_row[f'{side}_last_3_winrate'] = last_3_wr
    ml_input_row[f'{side}_last_5_winrate'] = last_5_wr
    ml_input_row[f'{side}_last_3_avg_score_for'] = last_3_avg_for
    ml_input_row[f'{side}_last_3_avg_score_against'] = last_3_avg_against
    ml_input_row[f'{side}_current_streak'] = streak

    for key in historical_h2h[side].keys():
        value = historical_h2h[side][key]
        ml_input_row[f'{side}_{key}'] = value
        print(f"  {side}_{key}: {value:.2f}")

# Difference features
if 'home_last_3_winrate' in ml_input_row and 'away_last_3_winrate' in ml_input_row:
    diff_last3_wr = ml_input_row['home_last_3_winrate'] - ml_input_row['away_last_3_winrate']
    ml_input_row['diff_last_3_winrate'] = diff_last3_wr
    print(f"\n📊 Difference features:")
    print(f"  Diff last 3 WR:    {diff_last3_wr:+.3f}")

In [ ]:
# 11. RANKINGS ÉS EGYÉB FEATURE-ÖK
print("\n" + "="*60)
print("9️⃣ RANKINGS ÉS EGYÉB FEATURE-ÖK")
print("="*60)

# Basic match info
ml_input_row['match_id'] = selected_match['match_id']
ml_input_row['event_id'] = TEST_EVENT_ID
ml_input_row['date'] = selected_match['date_parsed']
ml_input_row['match_url'] = selected_match['link']


# Teams
ml_input_row['team_home'] = teams['home']['team_name']
ml_input_row['team_away'] = teams['away']['team_name']

# Rankings
ml_input_row['home_current_rank'] = rank_data['home']['rank']
ml_input_row['away_current_rank'] = rank_data['away']['rank'] 
ml_input_row['home_rank_change'] = rank_data['home']['rank_change']
ml_input_row['away_rank_change'] = rank_data['away']['rank_change']

print(f"  Home rank: #{ml_input_row['home_current_rank']} (change: {ml_input_row['home_rank_change']:+d})")
print(f"  Away rank: #{ml_input_row['away_current_rank']} (change: {ml_input_row['away_rank_change']:+d})")

# Date features
ml_input_row['date_month'] = selected_match['date_parsed'].strftime("%m")
ml_input_row['date_day'] = int(selected_match['date_parsed'].strftime("%w")) + 1

# H2H features
ml_input_row['H2H_winrate_team1'] = match_h2h['home_win_rate']
ml_input_row['H2H_games'] = match_h2h['wins_home'] + match_h2h['wins_away']

# Match info
ml_input_row['match_rounds'] = selected_match['rounds']

# Label (score)
ml_input_row['score_home'] = selected_match['score_home']
ml_input_row['score_away'] = selected_match['score_away']
ml_input_row['label_home_win'] = 1 if selected_match['score_home'] > selected_match['score_away'] else 0

print("✅ Feature-ök összegyűjtve!")

In [ ]:
# 12. VÉGEREDMÉNY - ML INPUT ROW
print("\n" + "="*60)
print("🔚 VÉGEREDMÉNY - ML INPUT ROW")
print("="*60)

# DataFrame-ként megjelenítés
ml_df = pd.DataFrame([ml_input_row])

print("📊 DataFrame nézet:")
display(ml_df.T.style.set_caption("ML Input Row - Transposed"))

# Other

## Rankings history

In [ ]:
# Rankings history

# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings(url):
    driver = webdriver.Chrome()
    driver.get(url)
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    url_prev = driver.find_element(By.CLASS_NAME, "pagination-prev ").get_attribute("href")

    date_raw = driver.find_element(By.CLASS_NAME, "regional-ranking-header-text").text
    date = date_raw.split("ranking on ")[-1]
    # ordinal ragok eltávolítása
    date_clean = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date)
    # dátummá alakítás
    date_parsed = pd.to_datetime(date_clean, format="%B %d, %Y")

    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "").replace(" points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            profile_link = rank_div.find_element(By.CLASS_NAME, "moreLink").get_attribute("href")
            
            rankings.append({
                'date': date_parsed,
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points),
                'profile_link': profile_link
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings), url_prev

# Összes rankings leszedése
all_rankings = pd.DataFrame()
url = "https://www.hltv.org/ranking/teams/"
url_prev = ""
while url_prev != "https://www.hltv.org/ranking/teams/2024/december/30":
    rankings, url_prev = scrape_team_rankings(url)
    print(f"Scraping: {url}")
    all_rankings = pd.concat([all_rankings, rankings], ignore_index=True)
    display(rankings.sample(3))
    print(f"+{len(rankings)} (Total: {len(all_rankings)})")
    url = url_prev

In [ ]:
# Save rankings history

all_rankings.to_csv("data/all_rankings2025.csv", index=False)

## OddsPortal

In [ ]:
# Tournaments

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_oddsportal_cs_links():
    url = "https://www.oddsportal.com/results/#esports"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    # várjuk, amíg betöltődik legalább 1 tournament link
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")))

    time.sleep(2)  # kis extra wait, hogy minden JS lefusson

    links = driver.find_elements(By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")

    data = []
    for link in links:
        try:
            text = link.text.strip()
            href = link.get_attribute("href")
            if text.startswith("Counter-Strike "):
                name = text.replace("Counter-Strike ", "").strip()
                data.append({
                    "Name": name,
                    "url": href
                })
        except Exception as e:
            print(f"⚠️ Hiba egy link feldolgozásánál: {e}")
            continue

    driver.quit()
    df = pd.DataFrame(data)
    print(f"✅ {len(df)} Counter-Strike esemény található az OddsPortalon.")
    return df


# --- Példa futtatás ---
df_oddsportal = scrape_oddsportal_cs_links()
display(df_oddsportal.sample(5))


In [ ]:
# Search tournaments

#df_oddsportal[df_oddsportal.Name == "ESL Pro League Season 21"]["url"].iloc[0]
#df_oddsportal[df_oddsportal.Name.str.contains("IEM")].loc[196, "url"]

In [ ]:
# Odds of tournaments

tours = {'8292': {'name': 'ESL Pro League Season 21',
                  'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-esl-pro-league-season-21/results/'},
         '7907': {'name': 'BLAST Open London 2025',
                  'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-blast-open/results/'},
         '8038': {'name': 'IEM Cologne 2025',
              'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-intel-extreme-masters-cologne/results/'},
         '8036': {'name': 'IEM Melbourne 2025',
              'url': 'https://www.oddsportal.com/esports/counter-strike/counter-strike-iem-melbourne/results/'}
        }


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re, time
from datetime import datetime

def scrape_oddsportal_fixed(event_url, headless=False):
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=Mozilla/5.0")
    
    driver = webdriver.Chrome(options=opts)
    driver.get(event_url)
    wait = WebDriverWait(driver, 20)
    
    all_data = []
    page = 1
    
    print(f"Scrape: {event_url}")
    while True:
        print(f"🔍 Oldal {page} feldolgozása...")
        
        # Scroll, hogy minden betöltsön
        last_height = driver.execute_script("return document.body.scrollHeight")
        for _ in range(5):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2.5)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        time.sleep(2)
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.eventRow")))

        event_rows = driver.find_elements(By.CSS_SELECTOR, "div.eventRow")
        current_date = None

        for event in event_rows:
            # dátum keresése
            date_found = False
            
            # Dátum keresése az event teljes szövegében
            date_text = event.text.strip()
            if any(month in date_text for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                lines = date_text.split('\n')
                for line in lines:
                    if any(month in line for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                        # Dátum formázása - eltávolítjuk a " -" utáni részt
                        clean_date = line.split(' -')[0].strip()
                        current_date = clean_date
                        
                        # Dátum átalakítása YYYY-MM-dd formátumba
                        try:
                            date_obj = datetime.strptime(current_date, '%d %b %Y')
                            current_date = date_obj.strftime('%Y-%m-%d')
                        except ValueError:
                            # Ha nem sikerül átalakítani, marad az eredeti
                            pass
                        
                        date_found = True
                        break
            
            # Ha dátum sor, akkor tovább
            if date_found:
                continue

            # Ha nincs dátum, de van game-row, akkor meccs sor
            try:
                game_row = event.find_element(By.CSS_SELECTOR, "div[data-testid='game-row']")
            except:
                continue

            # csapatnevek
            participants = game_row.find_elements(By.CSS_SELECTOR, "a[title]")
            if len(participants) < 2:
                continue

            home_team = participants[0].get_attribute("title").strip()
            away_team = participants[1].get_attribute("title").strip()

            # oddsok
            odds_blocks = event.find_elements(By.CSS_SELECTOR, "div[data-testid^='odd-container'] p")
            odds = []
            for p in odds_blocks:
                try:
                    odds_text = p.text.strip().replace(",", ".")
                    if re.match(r"^\d+(\.\d+)?$", odds_text):
                        odds.append(float(odds_text))
                except:
                    continue
            
            home_odds = odds[0] if len(odds) > 0 else None
            away_odds = odds[1] if len(odds) > 1 else None

            all_data.append({
                "Date": current_date,
                "home_team": home_team,
                "away_team": away_team,
                "home_odds": home_odds,
                "away_odds": away_odds
            })

        # Következő oldal ellenőrzése
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, f"a.pagination-link[data-number='{page + 1}']")
            if next_button.is_enabled():
                print(f"➡️ Következő oldal: {page + 1}")
                driver.execute_script("arguments[0].click();", next_button)
                page += 1
                time.sleep(3)  # Várakozás az oldal betöltésére
                continue
            else:
                break
        except:
            # Ha nincs következő oldal, kilépünk
            break

    driver.quit()
    
    df = pd.DataFrame(all_data).drop_duplicates(subset=["home_team","away_team","home_odds","away_odds"])
    print(f"✅ Összesen {len(df)} meccs feldolgozva {page} oldalról.")
    return df

# Futtatás
df_odds = pd.DataFrame()
for id in tours.keys():
    print(f"Scraping {tours[id]['name']}")
    url_tour = tours[id]['url']
    df_odds_tour = scrape_oddsportal_fixed(url_tour, headless=False)
    df_odds_tour.Date = pd.to_datetime(df_odds_tour.Date)
    df_odds_tour['event_id'] = id
    display(df_odds_tour.sample(5))
    
    df_odds = pd.concat([df_odds, df_odds_tour], ignore_index=True)

print(f"Total odds found {len(df_odds)}")


In [ ]:
# Fuzzy matching

import pandas as pd
from fuzzywuzzy import process, fuzz
import json
from pathlib import Path

def fuzzy_match_teams(odds_teams, ranking_teams, save_path="fuzzy_mapping.json", threshold=70):
    """
    Fuzzy matching odds csapatok → rankings csapatok
    
    Args:
        odds_teams: list of team names from odds data
        ranking_teams: list of team names from rankings data  
        save_path: path to save the mapping
        threshold: minimum similarity score (0-100)
    """
    
    # Betöltés korábbi mapping-ből, ha létezik
    mapping = {}
    if Path(save_path).exists():
        with open(save_path, 'r', encoding='utf-8') as f:
            mapping = json.load(f)
        print(f"✅ Korábbi mapping betöltve: {len(mapping)} csapat")
    
    # Csak azokat a csapatokat dolgozzuk fel, amik még nincsenek a mapping-ben
    teams_to_process = [team for team in odds_teams if team not in mapping]
    
    if not teams_to_process:
        print("✅ Minden csapat már mapped!")
        return mapping
    
    print(f"🔍 {len(teams_to_process)} csapat feldolgozása...")
    
    for i, team in enumerate(teams_to_process, 1):
        print(f"\n[{i}/{len(teams_to_process)}] {team}")
        
        # Pontos egyezés
        if team in ranking_teams:
            mapping[team] = team
            print(f"   ✅ Auto match: {team}")
            continue
        
        # Fuzzy matching
        matches = process.extract(team, ranking_teams, limit=5, scorer=fuzz.token_sort_ratio)
        
        # Szűrés threshold alapján
        good_matches = [(match, score) for match, score in matches if score >= threshold]
        
        if len(good_matches) == 1:
            # Egyértelmű match
            best_match, best_score = good_matches[0]
            mapping[team] = best_match
            print(f"   ✅ Auto fuzzy: {team} → {best_match} ({best_score})")
        
        elif len(good_matches) > 1:
            # Több jó match - user dönt
            best_match, best_score = good_matches[0]
            second_match, second_score = good_matches[1]
            
            # Egyértelmű, ha nagy a különbség
            if best_score - second_score >= 15:
                mapping[team] = best_match
                print(f"   ✅ Clear winner: {team} → {best_match} ({best_score})")
            else:
                # User választ
                print(f"   ❓ Több hasonló találat:")
                for j, (match, score) in enumerate(good_matches, 1):
                    print(f"      {j}. {match} ({score})")
                print(f"      0. SKIP (kézi feldolgozás később)")
                
                try:
                    choice = input(f"   Válassz [1-{len(good_matches)}]: ").strip()
                    if choice.isdigit():
                        choice_int = int(choice)
                        if 1 <= choice_int <= len(good_matches):
                            chosen_match = good_matches[choice_int - 1][0]
                            mapping[team] = chosen_match
                            print(f"   👉 Kiválasztva: {team} → {chosen_match}")
                        else:
                            mapping[team] = None
                            print(f"   ⏭️ Skipped: {team}")
                    else:
                        mapping[team] = None
                        print(f"   ⏭️ Skipped: {team}")
                except:
                    mapping[team] = None
                    print(f"   ⏭️ Skipped: {team}")
        else:
            # Nincs jó match
            mapping[team] = None
            print(f"   ❌ No good match found for: {team}")
    
    # Mentés
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(mapping, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Mapping mentve: {save_path}")
    print(f"📊 Statisztika:")
    print(f"   Összes csapat: {len(mapping)}")
    print(f"   Sikeres match: {sum(1 for v in mapping.values() if v is not None)}")
    print(f"   Nincs match: {sum(1 for v in mapping.values() if v is None)}")
    
    return mapping

# Használat:
def apply_fuzzy_mapping(df_odds, rankings, mapping_path="fuzzy_mapping.json"):
    """Apply fuzzy mapping to merge odds with rankings"""
    
    # Betöltés mapping
    with open(mapping_path, 'r', encoding='utf-8') as f:
        mapping = json.load(f)
    
    # Csapatok összekapcsolása
    df_odds['home_team_mapped'] = df_odds['home_team'].map(mapping)
    df_odds['away_team_mapped'] = df_odds['away_team'].map(mapping)
    
    # Rankings merge
    rankings_clean = rankings.drop_duplicates('team_name')
    
    df_merged = df_odds.merge(
        rankings_clean, 
        left_on='home_team_mapped', 
        right_on='team_name', 
        how='left',
        suffixes=('', '_home')
    ).merge(
        rankings_clean,
        left_on='away_team_mapped', 
        right_on='team_name', 
        how='left',
        suffixes=('_home', '_away')
    )
    
    # Statisztika
    matched_home = df_merged['home_team_mapped'].notna().sum()
    matched_away = df_merged['away_team_mapped'].notna().sum()
    total_matches = len(df_merged)
    
    print(f"📊 Merge statisztika:")
    print(f"   Home team matched: {matched_home}/{total_matches} ({matched_home/total_matches*100:.1f}%)")
    print(f"   Away team matched: {matched_away}/{total_matches} ({matched_away/total_matches*100:.1f}%)")
    
    return df_merged

# PÉLDA HASZNÁLAT:
if __name__ == "__main__":
    # 2. Csapatok listázása
    odds_teams = pd.unique([*df_odds.home_team.unique(), *df_odds.away_team.unique()])
    ranking_teams = rankings.team_name.unique()
    
    print(f"📊 Adatok:")
    print(f"   Odds csapatok: {len(odds_teams)}")
    print(f"   Ranking csapatok: {len(ranking_teams)}")
    
    # 3. Fuzzy matching futtatása
    mapping = fuzzy_match_teams(odds_teams, ranking_teams, "team_mapping.json")
    
    # 4. Merge elvégzése
    df_final = apply_fuzzy_mapping(df_odds, rankings, "team_mapping.json")

In [ ]:
# Manual search
for t in rankings.team_name.unique():
    if 'nrg' in t.lower():
        print(t)

In [ ]:
# Check if names in dataframes

with open("team_mapping.json", 'r', encoding='utf-8') as f:
    mapping = json.load(f)

for t_op, t_hltv in mapping.items():
    if (t_op in odds_teams) & (t_hltv in rankings.team_name.unique()):
        pass
    else:
        print(f"Error for {t_op}/{t_hltv}")

In [ ]:
(df_odds.Date >= date-timedelta(days=1)) & (df_odds.Date <= date+timedelta(days=1))